# Stage 4 – Text Classification with RNN / LSTM / GRU

**Tasks covered**: 4-1 (data exploration) · 4-2 (RNN model) · 4-3 (training + evaluation) · 4-5 (LSTM & GRU)

**Dataset**: IMDB Movie Reviews — 25,000 train / 25,000 test, binary sentiment  
**Models**: Vanilla RNN · LSTM · GRU  

---
Run all cells top-to-bottom. On CPU set `USE_SUBSET = True` (~5 min total).  
On Google Colab with GPU set `USE_SUBSET = False` for full 25 K training.


In [ ]:
# Uncomment on Google Colab if packages are missing:
# !pip install torch scikit-learn matplotlib seaborn


In [ ]:
# ── 1. Imports & Device ────────────────────────────────────────────────────────
import os, re, sys, time, random, warnings
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Detect environment ─────────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
    print("Running on Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally")

# ── Device selection: CUDA > MPS (Apple Silicon) > CPU ────────────────────────
if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# ── 2. Configuration ───────────────────────────────────────────────────────────
# ┌──────────────────────────────────────────────────────────────────────────────┐
# │  USE_SUBSET = True  → 2500 reviews/class (5 000 total) — fast CPU training  │
# │  USE_SUBSET = False → full 12 500/class (25 000 total) — GPU recommended    │
# └──────────────────────────────────────────────────────────────────────────────┘
USE_SUBSET  = True    # ← flip to False on Colab/GPU
SUBSET_SIZE = 2500    # reviews per class when USE_SUBSET = True

# ── Data paths ─────────────────────────────────────────────────────────────────
if IN_COLAB:
    from google.colab import drive
    import subprocess
    drive.mount('/content/drive')
    # ── Set this to the path of stage_4_data.zip on your Drive ────────────────
    DRIVE_ZIP_PATH = '/content/drive/MyDrive/stage_4_data.zip'  # ← update if yours is in a subfolder
    # ── Unzip once into /content/ (skipped if already done) ───────────────────
    if not os.path.exists('/content/stage_4_data'):
        print(f'Unzipping {DRIVE_ZIP_PATH} ...')
        subprocess.run(['unzip', '-q', DRIVE_ZIP_PATH, '-d', '/content/'], check=True)
        print('Done.')
    BASE_DATA = '/content/stage_4_data'
else:
    BASE_DATA = '/Users/kavosh/Desktop/stage4/stage_4_data/stage_4_data'

TRAIN_DIR = os.path.join(BASE_DATA, 'text_classification', 'train')
TEST_DIR  = os.path.join(BASE_DATA, 'text_classification', 'test')

# ── Model hyper-parameters ─────────────────────────────────────────────────────
VOCAB_SIZE   = 10_000
EMBED_DIM    = 128
HIDDEN_DIM   = 256
N_LAYERS     = 2
DROPOUT      = 0.3
MAX_SEQ_LEN  = 200
BATCH_SIZE   = 64
N_EPOCHS     = 15
LR           = 1e-3

print("Configuration:")
print(f"  USE_SUBSET={USE_SUBSET}, SUBSET_SIZE={SUBSET_SIZE}")
print(f"  VOCAB={VOCAB_SIZE}, EMBED={EMBED_DIM}, HIDDEN={HIDDEN_DIM}")
print(f"  LAYERS={N_LAYERS}, DROPOUT={DROPOUT}, SEQ_LEN={MAX_SEQ_LEN}")
print(f"  BATCH={BATCH_SIZE}, EPOCHS={N_EPOCHS}, LR={LR}")


In [ ]:
# ── 3. Data Loading & Exploration (Task 4-1) ───────────────────────────────────
def load_split(split_dir, subset_size=None):
    """Load reviews from pos/ and neg/ subdirectories.
    Returns (texts, labels) where label 0=negative, 1=positive.
    """
    texts, labels = [], []
    for label_idx, cls in enumerate(['neg', 'pos']):
        folder = os.path.join(split_dir, cls)
        fnames = sorted(f for f in os.listdir(folder) if f.endswith('.txt'))
        if subset_size:
            fnames = fnames[:subset_size]
        for fname in fnames:
            with open(os.path.join(folder, fname), 'r',
                      encoding='utf-8', errors='replace') as fh:
                texts.append(fh.read().strip())
            labels.append(label_idx)
    return texts, labels

subset = SUBSET_SIZE if USE_SUBSET else None
train_texts, train_labels = load_split(TRAIN_DIR, subset)
test_texts,  test_labels  = load_split(TEST_DIR,  subset)

print(f"Train: {len(train_texts):,} reviews  "
      f"(neg={train_labels.count(0):,} | pos={train_labels.count(1):,})")
print(f"Test : {len(test_texts):,}  reviews  "
      f"(neg={test_labels.count(0):,} | pos={test_labels.count(1):,})")

# ── Peek at sample reviews ─────────────────────────────────────────────────────
print("\n── NEGATIVE sample ────────────────────────────────────────────────────")
print(train_texts[0][:400])
print("\n── POSITIVE sample ────────────────────────────────────────────────────")
pos_idx = next(i for i, l in enumerate(train_labels) if l == 1)
print(train_texts[pos_idx][:400])

# ── Length distribution ────────────────────────────────────────────────────────
lengths = [len(t.split()) for t in train_texts]
print(f"\nReview lengths: mean={np.mean(lengths):.0f}, "
      f"median={np.median(lengths):.0f}, max={max(lengths)}, min={min(lengths)}")


In [ ]:
# ── 4. Tokenisation & Vocabulary ───────────────────────────────────────────────
PAD, UNK = '<PAD>', '<UNK>'   # indices 0 and 1

def tokenize(text):
    """Lowercase, strip HTML tags, keep alphanumeric only, split on whitespace."""
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)           # strip HTML
    text = re.sub(r'[^a-z0-9\s]', ' ', text)      # keep letters + digits
    return text.split()

def build_vocab(texts, max_size=10_000):
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))
    vocab = {PAD: 0, UNK: 1}
    for word, _ in counter.most_common(max_size - 2):
        vocab[word] = len(vocab)
    return vocab

vocab     = build_vocab(train_texts, VOCAB_SIZE)
idx2word  = {v: k for k, v in vocab.items()}
print(f"Vocabulary size: {len(vocab):,}")
print(f"Top 15 words:    {list(vocab.keys())[2:17]}")


In [ ]:
# ── 5. Dataset & DataLoader ────────────────────────────────────────────────────
def encode(text, vocab, max_len):
    ids = [vocab.get(t, 1) for t in tokenize(text)[:max_len]]   # 1 = UNK
    ids += [0] * (max_len - len(ids))                            # pad
    return ids

class ReviewDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len):
        self.X = torch.tensor([encode(t, vocab, max_len) for t in texts],
                              dtype=torch.long)
        self.y = torch.tensor(labels, dtype=torch.float)
    def __len__(self):  return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

pin = device.type == 'cuda'
train_ds = ReviewDataset(train_texts, train_labels, vocab, MAX_SEQ_LEN)
test_ds  = ReviewDataset(test_texts,  test_labels,  vocab, MAX_SEQ_LEN)
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  pin_memory=pin)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, pin_memory=pin)

print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")


In [ ]:
# ── 6. RNN Model Architecture (Task 4-2) ──────────────────────────────────────
class TextRNN(nn.Module):
    """
    Unified text classifier supporting RNN / LSTM / GRU.

    Architecture:
        Embedding(vocab, embed_dim)
        → Dropout
        → RNN (unidirectional) / BiLSTM / BiGRU  (n_layers, hidden_dim)
        → Dropout on last hidden state
        → Linear(hidden_dim * [1 or 2], 1)   # *2 for bidirectional LSTM/GRU

    LSTM and GRU use bidirectional=True for better accuracy on long reviews.
    Binary cross-entropy with logits loss is used externally.
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers,
                 model_type='RNN', dropout=0.3, pad_idx=0, bidirectional=False):
        super().__init__()
        self.model_type    = model_type
        self.bidirectional = bidirectional

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.drop       = nn.Dropout(dropout)

        rnn_drop = dropout if n_layers > 1 else 0.0   # PyTorch requires 0 for 1-layer
        if model_type == 'RNN':
            self.rnn = nn.RNN(embed_dim, hidden_dim, n_layers,
                              batch_first=True, dropout=rnn_drop, nonlinearity='tanh',
                              bidirectional=bidirectional)
        elif model_type == 'LSTM':
            self.rnn = nn.LSTM(embed_dim, hidden_dim, n_layers,
                               batch_first=True, dropout=rnn_drop,
                               bidirectional=bidirectional)
        elif model_type == 'GRU':
            self.rnn = nn.GRU(embed_dim, hidden_dim, n_layers,
                              batch_first=True, dropout=rnn_drop,
                              bidirectional=bidirectional)
        else:
            raise ValueError(f"Unknown model_type: {model_type!r}")

        fc_in   = hidden_dim * 2 if bidirectional else hidden_dim
        self.fc = nn.Linear(fc_in, 1)

    def forward(self, x):
        # x : (batch, seq_len)
        emb = self.drop(self.embedding(x))     # (B, T, E)

        if self.model_type == 'LSTM':
            _, (h, _) = self.rnn(emb)          # h : (n_layers * dirs, B, H)
        else:
            _, h = self.rnn(emb)               # h : (n_layers * dirs, B, H)

        if self.bidirectional:
            # Concat last-layer forward + backward hidden states
            out = torch.cat([h[-2], h[-1]], dim=1)   # (B, H*2)
        else:
            out = h[-1]                              # (B, H)

        return self.fc(self.drop(out)).squeeze(-1)   # (B,)


# ── Architecture summary ───────────────────────────────────────────────────────
_demo = TextRNN(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, N_LAYERS, 'GRU', DROPOUT, bidirectional=True)
_x    = torch.randint(0, VOCAB_SIZE, (4, MAX_SEQ_LEN))
assert _demo(_x).shape == (4,), "Shape check failed"
n_params = sum(p.numel() for p in _demo.parameters() if p.requires_grad)
print(_demo)
print(f"\nTrainable parameters: {n_params:,}")
del _demo, _x


In [ ]:
# ── 7. Training & Evaluation Utilities ────────────────────────────────────────
criterion = nn.BCEWithLogitsLoss()

def run_epoch(model, loader, optimizer=None, training=True):
    """One full pass. Returns (avg_loss, accuracy, preds, labels)."""
    model.train() if training else model.eval()
    total_loss = total_correct = total_n = 0
    all_preds, all_labels = [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss   = criterion(logits, yb)
            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss    += loss.item() * len(yb)
            preds          = (torch.sigmoid(logits) >= 0.5).long()
            total_correct += (preds == yb.long()).sum().item()
            total_n       += len(yb)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(yb.long().cpu().tolist())

    return total_loss / total_n, total_correct / total_n, all_preds, all_labels


def train_model(model_type):
    """Train a TextRNN model and return (model, history).
    LSTM and GRU use bidirectional=True for higher accuracy on long reviews."""
    bidir = model_type in ('LSTM', 'GRU')
    model = TextRNN(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, N_LAYERS,
                    model_type, DROPOUT, bidirectional=bidir).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, patience=2, factor=0.5)

    history      = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    best_state   = None

    prefix = 'Bi' if bidir else '  '
    print(f"\n{'='*62}")
    print(f"  Training  {prefix}{model_type:<6}  |  "
          f"embed={EMBED_DIM}, hidden={HIDDEN_DIM}, layers={N_LAYERS}, bidirectional={bidir}")
    print(f"{'='*62}")
    print(f"{'Ep':>3} | {'Tr Loss':>8} | {'Tr Acc':>7} | "
          f"{'Val Loss':>9} | {'Val Acc':>8} | {'Time':>5}")
    print('-' * 50)

    for ep in range(1, N_EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_acc, _, _ = run_epoch(model, train_loader, optimizer)
        va_loss, va_acc, _, _ = run_epoch(model, test_loader,  training=False)
        scheduler.step(va_loss)

        history['train_loss'].append(tr_loss);  history['train_acc'].append(tr_acc)
        history['val_loss'].append(va_loss);    history['val_acc'].append(va_acc)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}

        print(f"{ep:3d} | {tr_loss:8.4f} | {tr_acc:7.4f} | "
              f"{va_loss:9.4f} | {va_acc:8.4f} | {time.time()-t0:4.1f}s")

    model.load_state_dict(best_state)
    print(f"\n  Best validation accuracy: {best_val_acc:.4f}")
    return model, history


In [ ]:
# ── 8. Train Vanilla RNN (Task 4-3) ───────────────────────────────────────────
rnn_model, rnn_history = train_model('RNN')


In [ ]:
# ── 9. Train LSTM (Task 4-5) ──────────────────────────────────────────────────
lstm_model, lstm_history = train_model('LSTM')


In [ ]:
# ── 10. Train GRU (Task 4-5) ──────────────────────────────────────────────────
gru_model, gru_history = train_model('GRU')


In [ ]:
# ── 11. Learning Curves (Task 4-3 / 4-5) ─────────────────────────────────────
histories = {'RNN': rnn_history, 'LSTM': lstm_history, 'GRU': gru_history}
palette   = {'RNN': '#e74c3c', 'LSTM': '#2ecc71', 'GRU': '#3498db'}
epochs    = range(1, N_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Learning Curves – Text Classification', fontsize=14, fontweight='bold')

for name, hist in histories.items():
    c = palette[name]
    ax1.plot(epochs, hist['train_loss'], '--', color=c, alpha=0.55, label=f'{name} train')
    ax1.plot(epochs, hist['val_loss'],   '-',  color=c, lw=2,       label=f'{name} val')
    ax2.plot(epochs, hist['train_acc'],  '--', color=c, alpha=0.55, label=f'{name} train')
    ax2.plot(epochs, hist['val_acc'],    '-',  color=c, lw=2,       label=f'{name} val')

for ax, title, ylabel in [(ax1, 'Loss', 'BCE Loss'), (ax2, 'Accuracy', 'Accuracy')]:
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch');  ax.set_ylabel(ylabel)
    ax.legend(fontsize=8, ncol=2);  ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(os.path.dirname(os.path.abspath('__file__')),
            'classification_learning_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: classification_learning_curves.png")


In [ ]:
# ── 12. Final Evaluation on Test Set (Task 4-3 / 4-5) ────────────────────────
print("\n" + "="*62)
print("  EVALUATION RESULTS ON TEST SET")
print("="*62)

results = {}
for name, model in [('RNN', rnn_model), ('LSTM', lstm_model), ('GRU', gru_model)]:
    _, acc, preds, labels = run_epoch(model, test_loader, training=False)
    f1  = f1_score(labels, preds, average='binary')
    results[name] = dict(accuracy=acc, f1=f1, preds=preds, labels=labels)
    print(f"\n─── {name} ──────────────────────────────────────────────────────")
    print(f"  Accuracy : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  F1 Score : {f1:.4f}")
    print(classification_report(labels, preds,
                                 target_names=['Negative', 'Positive'], digits=4))


In [ ]:
# ── 13. Confusion Matrices ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Confusion Matrices – Test Set', fontsize=14, fontweight='bold')

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(res['labels'], res['preds'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'], cbar=False)
    ax.set_title(f'{name}  (Acc={res["accuracy"]:.4f})', fontsize=11)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig(os.path.join(os.path.dirname(os.path.abspath('__file__')),
            'classification_confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: classification_confusion_matrices.png")


In [ ]:
# ── 14. Summary Table ─────────────────────────────────────────────────────────
print("\n" + "="*42)
print(f"{'Model':<8} | {'Accuracy':>10} | {'F1 Score':>10}")
print("-" * 42)
for name, res in results.items():
    print(f"{name:<8} | {res['accuracy']:10.4f} | {res['f1']:10.4f}")
print("="*42)

best_model = max(results, key=lambda k: results[k]['accuracy'])
print(f"\nBest model: {best_model} "
      f"(accuracy={results[best_model]['accuracy']:.4f})")
